In [1]:
# =========================================================
# Hallucination Detection Project (Kaggle Ready)
# - Input: knowledge + question + answer
# - Output: Supported (1) vs Hallucination (0)
# - Train/Test split: 70/30
# - Metrics: accuracy, precision, recall, f1
# - Model: microsoft/deberta-v3-base
# =========================================================

import os, json, random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

2026-02-01 19:38:27.616337: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769974707.779420      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769974707.825560      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769974708.218015      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769974708.218054      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769974708.218056      55 computation_placer.cc:177] computation placer alr

In [2]:
# ----------------------------
# 0) Reproducibility
# ----------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# ----------------------------
# 1) Load data
# ----------------------------
# Change this path for Kaggle:
# Example: "/kaggle/input/your-dataset/qa_data.json"
DATA_PATH = "/kaggle/input/qa-data-summarized-bart/qa_data_summarized_bart.json"   # <-- عدّلها لو لزم

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"File not found: {DATA_PATH}. Please upload it or set correct Kaggle path.")

def load_json_or_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)

        # Case 1: JSON array
        if first_char == "[":
            data = json.load(f)
        else:
            # Case 2: JSONL
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    return data

raw = load_json_or_jsonl(DATA_PATH)


# raw could be list[dict] or jsonl-like content; handle both safely
if isinstance(raw, dict):
    # in case it's wrapped
    # try typical keys:
    for k in ["data", "items", "records"]:
        if k in raw and isinstance(raw[k], list):
            raw = raw[k]
            break

if not isinstance(raw, list):
    raise ValueError("Expected a list of records in JSON.")

required_cols = {"knowledge", "question", "right_answer", "hallucinated_answer"}
for i, r in enumerate(raw[:5]):
    if not required_cols.issubset(set(r.keys())):
        raise ValueError(f"Record {i} missing keys. Required: {required_cols}. Found: {r.keys()}")

df = pd.DataFrame(raw)
df = df.dropna(subset=["knowledge", "question", "right_answer", "hallucinated_answer"]).reset_index(drop=True)

print("Loaded records:", len(df))
print(df.head(2))

Loaded records: 10000
                                           knowledge  \
0  Arthur's Magazine (1844–1846) was an American ...   
1  The Oberoi family is an Indian family that is ...   

                                            question       right_answer  \
0  Which magazine was started first Arthur's Maga...  Arthur's Magazine   
1  The Oberoi family is part of a hotel company t...              Delhi   

                                 hallucinated_answer  
0                 First for Women was started first.  
1  The Oberoi family's hotel company is based in ...  


In [4]:
print("Number of records:", len(raw))
print(raw[0])


Number of records: 10000
{'knowledge': "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. First for Women is a woman's magazine published by Bauer Media Group in the USA. It was first published in 1846.", 'question': "Which magazine was started first Arthur's Magazine or First for Women?", 'right_answer': "Arthur's Magazine", 'hallucinated_answer': 'First for Women was started first.'}


In [5]:
# ----------------------------
# 2) Build classification dataset
# ----------------------------
# Each original record -> 2 samples:
# - supported: right_answer => 1
# - hallucination: hallucinated_answer => 0
rows = []
for _, r in df.iterrows():
    rows.append({
        "knowledge": r["knowledge"],
        "question": r["question"],
        "answer": r["right_answer"],
        "label": 1
    })
    rows.append({
        "knowledge": r["knowledge"],
        "question": r["question"],
        "answer": r["hallucinated_answer"],
        "label": 0
    })

clf_df = pd.DataFrame(rows)
clf_df = clf_df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print("\nClassification samples:", len(clf_df))
print(clf_df["label"].value_counts())



Classification samples: 20000
label
1    10000
0    10000
Name: count, dtype: int64


In [6]:
# ----------------------------
# 3) Train/Test split 70/30 (stratified)
# ----------------------------
train_df, test_df = train_test_split(
    clf_df,
    test_size=0.30,
    random_state=SEED,
    stratify=clf_df["label"]
)

print("\nTrain size:", len(train_df), "Test size:", len(test_df))
print("Train label dist:\n", train_df["label"].value_counts(normalize=True))
print("Test label dist:\n", test_df["label"].value_counts(normalize=True))


Train size: 14000 Test size: 6000
Train label dist:
 label
0    0.5
1    0.5
Name: proportion, dtype: float64
Test label dist:
 label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [7]:
# ----------------------------
# 4) Tokenization setup
# ----------------------------
MODEL_NAME = "microsoft/deberta-v3-base"  # strong baseline
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_input_text(knowledge, question, answer):
    # Simple and effective formatting
    return f"KNOWLEDGE:\n{knowledge}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}"

class HallucinationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = build_input_text(row["knowledge"], row["question"], row["answer"])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length
        )
        enc["labels"] = int(row["label"])
        return enc

train_ds = HallucinationDataset(train_df, tokenizer)
test_ds  = HallucinationDataset(test_df, tokenizer)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [8]:
# ----------------------------
# 5) Model
# ----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# ----------------------------
# 6) Metrics
# ----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [10]:
# ----------------------------
# 7) TrainingArguments (Kaggle-friendly)
# ----------------------------
# Tips:
# - If GPU is available, this will be fast enough.
# - Adjust batch size if you get OOM.

args = TrainingArguments(
    output_dir="hallucination_detector_deberta",

    # 🔴 IMPORTANT (new transformers)
    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    do_train=True,
    do_eval=True,

    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_ratio=0.05,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none"
)

print("TrainingArguments fixed & ready ✅")


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)


TrainingArguments fixed & ready ✅


/tmp/ipykernel_55/554254211.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
# ----------------------------
# 8) Train
# ----------------------
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.127000,0.103562,0.977500,0.964645,0.991333,0.977807
2,0.075300,0.077442,0.979000,0.966558,0.992333,0.979276


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1750, training_loss=0.11463576303209577, metrics={'train_runtime': 938.6368, 'train_samples_per_second': 29.83, 'train_steps_per_second': 1.864, 'total_flos': 1942929098540928.0, 'train_loss': 0.11463576303209577, 'epoch': 2.0})

In [12]:
# ----------------------------
# 9) Final evaluation on 30% test
# ----------------------------
pred_out = trainer.predict(test_ds)
test_logits = pred_out.predictions
test_labels = pred_out.label_ids
test_preds = np.argmax(test_logits, axis=1)

acc = accuracy_score(test_labels, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="binary", pos_label=1, zero_division=0
)

print("\n=== TEST METRICS (30%) ===")
print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print("\n=== Classification Report ===")
print(classification_report(test_labels, test_preds, target_names=["hallucination(0)", "supported(1)"], zero_division=0))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(test_labels, test_preds))

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



=== TEST METRICS (30%) ===
Accuracy : 0.979
Precision: 0.9665584415584415
Recall   : 0.9923333333333333
F1       : 0.9792763157894737

=== Classification Report ===
                  precision    recall  f1-score   support

hallucination(0)       0.99      0.97      0.98      3000
    supported(1)       0.97      0.99      0.98      3000

        accuracy                           0.98      6000
       macro avg       0.98      0.98      0.98      6000
    weighted avg       0.98      0.98      0.98      6000


=== Confusion Matrix ===
[[2897  103]
 [  23 2977]]


In [13]:
import pandas as pd

# Save metrics
metrics = {
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv("test_metrics.csv", index=False)

# Save confusion matrix
cm_df = pd.DataFrame(
    confusion_matrix(test_labels, test_preds),
    columns=["Pred_Hallucination", "Pred_Supported"],
    index=["True_Hallucination", "True_Supported"]
)
cm_df.to_csv("confusion_matrix.csv")

# Save detailed predictions
test_df_out = test_df.copy()
test_df_out["prediction"] = test_preds
test_df_out["correct"] = (test_preds == test_labels)
test_df_out.to_csv("test_predictions.csv", index=False)

print("Results saved:")
print("- test_metrics.csv")
print("- confusion_matrix.csv")
print("- test_predictions.csv")


Results saved:
- test_metrics.csv
- confusion_matrix.csv
- test_predictions.csv


In [14]:
# ----------------------------
# 10) Save model for later inference (Kaggle output folder)
# ----------------------------
SAVE_DIR = "saved_hallucination_detector"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\nSaved to: {SAVE_DIR}")


Saved to: saved_hallucination_detector


In [15]:
# ----------------------------
# 11) Inference function (for later: generated answer detection)
# ----------------------------
@torch.inference_mode()
def detect_hallucination(knowledge, question, answer, threshold=0.5):
    """
    Returns:
      - supported_prob (probability of label=1)
      - predicted_label: 1 supported, 0 hallucination
    """
    text = build_input_text(knowledge, question, answer)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
    supported_prob = float(probs[1])
    pred = 1 if supported_prob >= threshold else 0
    return supported_prob, pred

# Example quick test:
ex = df.iloc[0]
p1, y1 = detect_hallucination(ex["knowledge"], ex["question"], ex["right_answer"])
p2, y2 = detect_hallucination(ex["knowledge"], ex["question"], ex["hallucinated_answer"])

print("\nExample check:")
print("Right answer => supported_prob:", p1, "pred:", y1)
print("Hallucinated answer => supported_prob:", p2, "pred:", y2)


Example check:
Right answer => supported_prob: 0.998779833316803 pred: 1
Hallucinated answer => supported_prob: 0.001448271214030683 pred: 0
